# Jour 2 · Baselines et métriques de prévision


## Objectifs

- construire une prévision naïve avant un modèle complexe
- calculer MAE et RMSE
- évaluer un horizon futur fixe sans fuite de données

Une **baseline** répond à la question : notre modèle fait-il mieux qu'une règle évidente ? Ici nous prévoyons les sept derniers jours à partir de ce qui était connu juste avant la frontière.

![Ingénieur observant l'historique des capteurs HVAC et sa prolongation sous forme de prévision](../assets/jour_02/00_prevision_hvac.png)

*Le forecasting utilise le passé connu pour estimer une zone future encore inconnue, puis compare cette estimation aux valeurs réellement observées.*

![Comparaison visuelle entre une baseline de persistance et une baseline saisonnière](../assets/jour_02/01_baseline_persistance_saisonniere.png)

*La persistance prolonge la dernière valeur ; la baseline saisonnière répète un motif récent. Ces règles simples servent de niveau minimal à battre.*

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error


from pathlib import Path


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "datasets").is_dir():
            return candidate
    raise FileNotFoundError("Dossier datasets introuvable. Lancez Jupyter depuis le projet.")


ROOT = find_project_root()
DATA_DIR = ROOT / "datasets"


plt.style.use("seaborn-v0_8-whitegrid")
df = pd.read_csv(DATA_DIR / "prepared" / "iot_hvac_clean.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
hourly = df.set_index("timestamp")["power_kw"].resample("1h").mean()

cutoff = hourly.index.max() - pd.Timedelta(days=7)
train = hourly.loc[hourly.index <= cutoff]
test = hourly.loc[hourly.index > cutoff]
print(len(train), "heures d'entraînement |", len(test), "heures de test")

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": math.sqrt(mean_squared_error(y_true, y_pred)),
    }


persistence = pd.Series(train.iloc[-1], index=test.index, name="persistance")

last_week_pattern = train.iloc[-24 * 7:].to_numpy()
seasonal_values = np.resize(last_week_pattern, len(test))
seasonal_naive = pd.Series(seasonal_values, index=test.index, name="semaine précédente")

scores = pd.DataFrame({
    "persistance": evaluate(test, persistence),
    "saisonnier": evaluate(test, seasonal_naive),
}).T
scores

- **MAE** : erreur absolue moyenne, facile à interpréter dans l'unité du signal.
- **RMSE** : pénalise davantage les grandes erreurs.

Plus la valeur est petite, meilleure est la prévision. Il faut toujours la comparer à l'échelle habituelle de la variable.

![Comparaison de la réaction de la MAE et de la RMSE à une grande erreur](../assets/jour_02/01_mae_et_rmse.png)

*La MAE reste directement interprétable ; la RMSE donne davantage de poids aux grandes erreurs.*

![Puissance réelle comparée à deux baselines sur la période de test](../assets/jour_02/01_baselines_sur_donnees_reelles.png)

*Les modèles sont comparés sur exactement le même futur : la courbe seule aide à comprendre, les métriques permettent de décider.*

In [ ]:
ax = test.plot(figsize=(13, 4), label="réel", color="black")
persistence.plot(ax=ax, label="persistance", alpha=0.8)
seasonal_naive.plot(ax=ax, label="baseline saisonnière", alpha=0.8)
ax.set_ylabel("kW")
ax.set_title("Prévoir les sept derniers jours")
ax.legend()
plt.show()

### À vous de jouer — calculer la MAE à la main

Recalculez la MAE de la baseline saisonnière uniquement avec les opérations Pandas, puis comparez-la à la valeur de scikit-learn.

**Indice :** MAE = moyenne de abs(valeur réelle - prévision).

In [ ]:
# Écrivez votre code ici.
pass

### À vous de jouer — essayer un motif journalier

Répétez les 24 dernières heures d'entraînement sur toute la semaine de test. Comparez ses métriques au motif hebdomadaire.

**Indice :** np.resize répète un tableau jusqu'à la longueur demandée.

In [ ]:
# Écrivez votre code ici.
pass

## À retenir

Une baseline saisonnière peut être redoutablement efficace. Un modèle plus complexe n'est utile que s'il apporte un gain mesurable ou une information supplémentaire.